# S?IT Aspect Sentiment Quad Prediction ? Offline Kaggle

This notebook trains and evaluates S?IT without internet access. Attach a prepared offline bundle, the project source, a local base model, and an optional wheelhouse through Kaggle Inputs. Stanza parses and model files are prepared before the Kaggle run.


In [ ]:
# ---- OFFLINE CONFIG ----
# Attach the prepared offline_bundle, project source, and base model as Kaggle Inputs.
GITHUB_REPO = None
PROJECT_DIR = None  # auto-detected from the attached kaggle_offline_s2it dataset
DATASET_INPUT_DIR = None
DATA_DIR = None
OFFLINE_WHEELHOUSE = None
OFFLINE_BUNDLE_DIR = None
OUTPUT_DIR = "/kaggle/working/s2it_outputs"
BASE_MODEL = "/kaggle/input/models/dipamc77/qwen2.5-7b-instruct/transformers/default/1/Qwen2.5-7B-Instruct"
OFFLINE_MODE = True
HF_CACHE_DIR = "/kaggle/working/hf_cache"
EPOCHS = 2
BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 5e-5
LORA_RANK = 8
MAX_LENGTH = 768
DATA_PERCENT = 5
DATA_SAMPLE_SEED = 444
PARSER_LANGUAGES = []
PARSER_LANGUAGE = "auto"
RUN_TRAINING = True
RUN_INFERENCE = True
MODEL_DIR = f"{OUTPUT_DIR}/final"
MAX_EVAL_RECORDS = None
MAX_NEW_TOKENS = 256
PARSER_USE_GPU = False

from pathlib import Path
from zipfile import ZipFile
import os, sys
input_root = Path("/kaggle/input")
upload_archive = next(input_root.rglob("kaggle_offline_upload.zip"), None) if input_root.exists() else None
if upload_archive is not None:
    extracted_root = Path("/kaggle/working/s2it_offline_upload")
    extracted_root.mkdir(parents=True, exist_ok=True)
    with ZipFile(upload_archive) as archive:
        archive.extractall(extracted_root)
    PROJECT_DIR = str(extracted_root / "project")
    DATA_DIR = str(extracted_root / "offline_bundle")
    OFFLINE_BUNDLE_DIR = DATA_DIR
    OFFLINE_WHEELHOUSE = str(Path(DATA_DIR) / "wheelhouse")
if PROJECT_DIR is None or DATA_DIR is None:
    parse_cache_candidates = list(input_root.rglob("stanza_parses.jsonl")) if input_root.exists() else []
    bundle_candidates = [p.parent for p in parse_cache_candidates
                         if p.parent.name == "offline_bundle"
                         and (p.parent / "train.jsonl").is_file()
                         and (p.parent.parent / "project" / "src" / "asqp_s2it").is_dir()]
    if not bundle_candidates:
        raise FileNotFoundError("Could not find attached Kaggle dataset containing project/ and offline_bundle/")
    DATA_DIR = str(bundle_candidates[0])
    OFFLINE_BUNDLE_DIR = DATA_DIR
    OFFLINE_WHEELHOUSE = str(Path(DATA_DIR) / "wheelhouse")
    PROJECT_DIR = str(Path(DATA_DIR).parent / "project")
if BASE_MODEL is None or not Path(BASE_MODEL).is_dir():
    model_configs = [p.parent for p in input_root.rglob("config.json")
                     if "qwen" in str(p.parent).lower()] if input_root.exists() else []
    if not model_configs:
        raise FileNotFoundError("Attach the local Qwen model as a Kaggle Input or set BASE_MODEL to its mounted path")
    BASE_MODEL = str(sorted(model_configs, key=lambda p: ("7b" not in str(p).lower(), len(str(p))))[0])
PROJECT_DIR = str(Path(PROJECT_DIR))
DATA_DIR = str(Path(DATA_DIR))
OFFLINE_BUNDLE_DIR = str(Path(OFFLINE_BUNDLE_DIR))
OFFLINE_WHEELHOUSE = str(Path(OFFLINE_WHEELHOUSE))
BASE_MODEL = str(Path(BASE_MODEL))
OUTPUT_DIR = str(Path(OUTPUT_DIR))
MODEL_DIR = str(Path(MODEL_DIR))
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HOME"] = HF_CACHE_DIR
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print(f"Offline source: {PROJECT_DIR}\nData/cache: {OFFLINE_BUNDLE_DIR}\nModel: {BASE_MODEL}")


## 1. Offline Kaggle inputs

Attach the `kaggle_offline_s2it` Dataset and the Qwen model under **Models**. The config cell finds `project/`, `offline_bundle/`, and the Qwen model automatically; set `BASE_MODEL` manually only if model discovery is ambiguous. No Internet access is needed.


In [ ]:
from pathlib import Path
import sys, subprocess

project_path = Path(PROJECT_DIR)
data_path = Path(DATA_DIR)
model_path = Path(BASE_MODEL)
for required in (project_path / "src" / "asqp_s2it", data_path / "train.jsonl", data_path / "dev.jsonl", data_path / "test.jsonl", Path(OFFLINE_BUNDLE_DIR) / "stanza_parses.jsonl", model_path / "config.json"):
    if not required.exists():
        raise FileNotFoundError(f"Missing offline Kaggle input: {required}")
sys.path.insert(0, str(project_path / "src"))
# Missing packages are installed exclusively from the attached local wheelhouse.
subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", "--find-links", OFFLINE_WHEELHOUSE,
                       "stanza==1.15.0", "peft==0.14.0", "transformers==4.48.3",
                       "tokenizers==0.21.0", "accelerate==1.15.0", "pandas==2.3.2",
                       "huggingface-hub==0.28.1", "safetensors==0.6.2",
                       "emoji==2.16.0", "networkx==3.7", "orjson==3.13.0",
                       "platformdirs==4.12.4", "protobuf==7.36.2", "regex==2026.9.29",
                       "udtools==0.2.8", "udapi==0.5.2", "termcolor==3.2.0"])

print("Offline project, dataset, parse cache, model, and local dependencies are ready.")


In [ ]:
# Imports only; dependencies must be present in the Kaggle image or bundled in its offline environment.
import json, time, pandas as pd
from IPython.display import display
from asqp_s2it.data import read_jsonl, category_labels, write_jsonl
from asqp_s2it.metrics import exact_quad_f1_by_instance
from asqp_s2it.model import S2ITPredictor
from asqp_s2it.syntax import CachedDependencyParser
from asqp_s2it.timing import format_duration
from asqp_s2it.training import train
import torch
print("PyTorch:", torch.__version__, "CUDA:", torch.cuda.is_available())


## 2. Äá»c dá»¯ liá»‡u vÃ  hiá»ƒn thá»‹ thá»‘ng kÃª

CÃ¡c nhÃ£n láº¥y tá»« `quads`: `aspect_term`, `aspect_category`, `opinion_term`, `sentiment`. `null` Ä‘Æ°á»£c giá»¯ nguyÃªn cho aspect/opinion implicit.

In [ ]:
# Load locally prepared, language-tagged splits. Parse cache contains all rows, so trial sampling is safe.
import random
splits = {}
for split_name in ("train", "dev", "test"):
    records = read_jsonl(Path(DATA_DIR) / f"{split_name}.jsonl")
    if DATA_PERCENT < 100:
        rng = random.Random(DATA_SAMPLE_SEED + ("train", "dev", "test").index(split_name))
        n = max(1, int(len(records) * DATA_PERCENT / 100 + 0.999999))
        records = [records[i] for i in sorted(rng.sample(range(len(records)), n))]
    splits[split_name] = records
train_records, dev_records, test_records = splits["train"], splits["dev"], splits["test"]
PARSER_LANGUAGES = sorted({r.language for rows in splits.values() for r in rows if r.language})
categories = category_labels(train_records)
split_table = pd.DataFrame([{"Split": k, "Sentences": len(v), "Languages": ", ".join(sorted({r.language for r in v if r.language})), "Quadruples": sum(len(r.quads) for r in v)} for k,v in splits.items()])
display(split_table)
category_table = pd.DataFrame([{"Aspect category": c, "Train quadruples": sum(q.aspect_category == c for r in train_records for q in r.quads), "Dev quadruples": sum(q.aspect_category == c for r in dev_records for q in r.quads), "Test quadruples": sum(q.aspect_category == c for r in test_records for q in r.quads)} for c in categories])
display(category_table)


## 3. Chuáº©n bá»‹ dependency parser

Notebook chá»‰ giá»¯ cÃ¡c language Ä‘Ã£ cho phÃ©p: tiáº¿ng Anh, Viá»‡t, Trung giáº£n thá»ƒ, Nháº­t, Äá»©c, ThÃ¡i vÃ  PhÃ¡p. Record ngoÃ i danh sÃ¡ch bá»‹ loáº¡i khá»i tá»«ng split vÃ  thá»‘ng kÃª riÃªng; do Ä‘Ã³ Stanza chá»‰ táº£i parser cho cÃ¡c language cÃ²n láº¡i. Chá»‰ táº£i cÃ¡c model `tokenize,pos,depparse` vÃ o `STANZA_CACHE_DIR`, khÃ´ng táº£i gÃ³i `default.zip` Ä‘áº§y Ä‘á»§. Pipeline dÃ¹ng lemmatizer identity Ä‘á»ƒ Ä‘Ã¡p á»©ng prerequisite báº¯t buá»™c cá»§a depparse mÃ  khÃ´ng cáº§n táº£i model lemma riÃªng; cache Hugging Face riÃªng cá»§a tá»«ng Stanza language Ä‘Æ°á»£c dá»n sau khi táº£i Ä‘á»ƒ giáº£m dung lÆ°á»£ng. Kaggle Internet pháº£i báº­t.

In [ ]:
parser_setup_started = time.perf_counter()
dependency_parser = CachedDependencyParser(Path(OFFLINE_BUNDLE_DIR) / "stanza_parses.jsonl")
parser_setup_seconds = time.perf_counter() - parser_setup_started
print(f"Loaded offline Stanza parse cache in {format_duration(parser_setup_seconds)}")


## 4. Huáº¥n luyá»‡n SÂ²IT

Ba giai Ä‘oáº¡n cháº¡y tuáº§n tá»± trÃªn cÃ¹ng LoRA adapter: **(1)** global syntax-guided extraction vá»›i tÃ¡c vá»¥ trÃ­ch xuáº¥t aspectâ†’opinion vÃ  opinionâ†’aspect; **(2)** local syntax-guided classification trÃªn cÃ¡c cáº·p gold aspectâ€“opinion; **(3)** fine-grained structural tuning gá»“m link prediction theo global dependency relations, node classification vÃ  phÃ¢n loáº¡i quadruple theo local one-hop subgraphs. Khi chuáº©n bá»‹ dá»¯ liá»‡u, log tiáº¿n Ä‘á»™ má»—i khoáº£ng 5% kÃ¨m elapsed/ETA; khi fine-tune, log step/total, %, epoch, loss, elapsed vÃ  ETA má»—i 10 steps. Thá»i gian tá»«ng giai Ä‘oáº¡n váº«n Ä‘Æ°á»£c ghi riÃªng vÃ  tá»•ng há»£p phÃ­a dÆ°á»›i.

In [ ]:
if RUN_TRAINING:
    notebook_training_started = time.perf_counter()
    training_metrics = train(
        train_instances=train_records,
        dev_instances=dev_records,
        output_dir=OUTPUT_DIR,
        model_name=BASE_MODEL,
        parser=dependency_parser,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE,
        lora_rank=LORA_RANK,
        max_length=MAX_LENGTH,
    )
    notebook_training_wall_seconds = time.perf_counter() - notebook_training_started
    training_metrics["notebook_training_wall_seconds"] = notebook_training_wall_seconds
    training_metrics["parser_setup_seconds"] = parser_setup_seconds
    training_metrics_path = Path(MODEL_DIR) / "training_metrics.json"
    training_metrics_path.write_text(json.dumps(training_metrics, ensure_ascii=False, indent=2), encoding="utf-8")
else:
    training_metrics_path = Path(MODEL_DIR) / "training_metrics.json"
    if training_metrics_path.is_file():
        training_metrics = json.loads(training_metrics_path.read_text(encoding="utf-8"))
    else:
        training_metrics = None
    print("Training skipped; loading existing adapter if available:", MODEL_DIR)

if training_metrics:
    stage_seconds = training_metrics.get("stage_training_seconds", {})
    timing_rows = [
        {"Phase": "Global syntax-guided extraction", "Seconds": stage_seconds.get("global_extraction", 0.0)},
        {"Phase": "Local syntax-guided classification", "Seconds": stage_seconds.get("local_classification", 0.0)},
        {"Phase": "Fine-grained structural tuning", "Seconds": stage_seconds.get("fine_grained_structure", 0.0)},
        {"Phase": "Total model training", "Seconds": training_metrics.get("training_seconds", 0.0)},
        {"Phase": "End-to-end training (incl. prep/load/save)", "Seconds": training_metrics.get("total_elapsed_seconds", 0.0)},
        {"Phase": "Notebook training call wall time", "Seconds": training_metrics.get("notebook_training_wall_seconds", training_metrics.get("total_elapsed_seconds", 0.0))},
    ]
    timing_table = pd.DataFrame(timing_rows)
    timing_table["Time (HH:MM:SS)"] = timing_table["Seconds"].map(format_duration)
    timing_table["Seconds"] = timing_table["Seconds"].round(2)
    display(timing_table)
    print("Training time:", training_metrics.get("training_time", "n/a"))
    print("End-to-end:", training_metrics.get("total_elapsed_time", "n/a"))
    training_table_path = Path(OUTPUT_DIR) / "training_time.csv"
    timing_table.to_csv(training_table_path, index=False)
    print("Saved:", training_table_path)

## 5. Dá»± Ä‘oÃ¡n vÃ  exact-match F1

Theo pháº§n inference cá»§a paper, bÆ°á»›c trÃ­ch xuáº¥t giáº£i mÃ£ hai chiá»u aspectâ†’opinion vÃ  opinionâ†’aspect báº±ng beam search kÃ­ch thÆ°á»›c 4; káº¿t quáº£ hai hÆ°á»›ng Ä‘Æ°á»£c gá»™p, sau Ä‘Ã³ chuyá»ƒn sang phÃ¢n loáº¡i báº±ng local one-hop subgraphs. F1 micro chá»‰ tÃ­nh quadruple Ä‘Ãºng khi cáº£ bá»‘n trÆ°á»ng khá»›p vÃ  thuá»™c Ä‘Ãºng instance; khÃ´ng tÃ­nh partial match. Äáº·t `MAX_EVAL_RECORDS` thÃ nh sá»‘ nhá» hÆ¡n Ä‘á»ƒ cháº¡y thá»­ nhanh.

In [ ]:
if RUN_INFERENCE:
    evaluation_records = test_records if MAX_EVAL_RECORDS is None else test_records[:MAX_EVAL_RECORDS]
    if len(evaluation_records) != len(test_records):
        print(f"Quick evaluation: {len(evaluation_records)} / {len(test_records)} test records")

    predictor = S2ITPredictor(MODEL_DIR, dependency_parser)
    prediction_rows = []
    inference_started = time.perf_counter()
    for index, instance in enumerate(evaluation_records, start=1):
        predicted_quads = predictor.predict(instance, categories, max_new_tokens=MAX_NEW_TOKENS)
        prediction_rows.append({
            "instance_id": instance.instance_id,
            "text": instance.text,
            "language": instance.language,
            "quadruples": predicted_quads,
        })
        if index % 100 == 0 or index == len(evaluation_records):
            print(f"Predicted {index}/{len(evaluation_records)}")
    inference_seconds = time.perf_counter() - inference_started

    prediction_path = Path(OUTPUT_DIR) / "test_predictions.jsonl"
    write_jsonl(prediction_path, prediction_rows)
    gold_by_id = {row.instance_id: row.quads for row in evaluation_records}
    predictions_by_id = {row["instance_id"]: row["quadruples"] for row in prediction_rows}
    evaluation = exact_quad_f1_by_instance(gold_by_id, predictions_by_id)

    result_rows = [
        {"Metric": "Precision", "Value": evaluation["precision"]},
        {"Metric": "Recall", "Value": evaluation["recall"]},
        {"Metric": "Exact quadruple micro-F1", "Value": evaluation["f1"]},
        {"Metric": "True positive quadruples", "Value": evaluation["true_positives"]},
        {"Metric": "Predicted quadruples", "Value": evaluation["predicted"]},
        {"Metric": "Gold quadruples", "Value": evaluation["gold"]},
        {"Metric": "Inference time (HH:MM:SS)", "Value": format_duration(inference_seconds)},
        {"Metric": "Inference time (seconds)", "Value": round(inference_seconds, 2)},
    ]
    if training_metrics:
        result_rows.extend([
            {"Metric": "Training time (HH:MM:SS)", "Value": training_metrics.get("training_time", "n/a")},
            {"Metric": "Training time (seconds)", "Value": round(training_metrics.get("training_seconds", 0.0), 2)},
            {"Metric": "End-to-end training (HH:MM:SS)", "Value": training_metrics.get("total_elapsed_time", "n/a")},
        ])
    results_table = pd.DataFrame(result_rows)
    display(results_table)

    results_csv_path = Path(OUTPUT_DIR) / "evaluation_metrics.csv"
    results_table.to_csv(results_csv_path, index=False)
    print(f"Predictions: {prediction_path}")
    print(f"Metrics CSV: {results_csv_path}")
else:
    print("Inference skipped. Set RUN_INFERENCE=True to predict/evaluate.")

## CÃ¡c file output

- `s2it_outputs/final/`: LoRA adapter, tokenizer, danh sÃ¡ch category vÃ  `training_metrics.json`.
- `s2it_outputs/training_time.csv`: thá»i gian tá»«ng stage vÃ  tá»•ng thá»i gian.
- `s2it_outputs/test_predictions.jsonl`: quadruple dá»± Ä‘oÃ¡n theo instance.
- `s2it_outputs/evaluation_metrics.csv`: precision, recall, exact quadruple F1 vÃ  thá»i gian inference/training.

Táº£i cÃ¡c file tá»« má»¥c **Output** cá»§a Kaggle notebook sau khi cháº¡y xong.